## Практика. Нормализация и кодирование данных

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning) 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_style("whitegrid")

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler


from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import confusion_matrix
from sklearn.metrics import RocCurveDisplay

>#### Задание 1
Прочитайте файл `car_repair`.

In [2]:
df = pd.read_csv('data/car_repair.csv')
df 

,damage_zone,car_class,detail_code,car_age,engine_volume,mileage,repair_cost
0,задняя,эконом,WHEEL-01,6,4.9,65.4,41.3
1,боковая,эконом,BUMPER-03,19,4.1,13.5,89.0
2,боковая,премиум,PAINT-02,23,4.0,56.4,155.3
3,задняя,эконом,SUSP-01,22,5.2,243.5,263.3
4,передняя,эконом,MOTOR-01,6,3.7,71.7,600.0
...,...,...,...,...,...,...,...
1995,задняя,комфорт,BUMPER-03,0,5.1,244.6,136.4
1996,боковая,эконом,LIGHT-01,9,3.5,311.6,37.2
1997,передняя,эконом,BUMPER-03,1,1.7,287.6,106.7
1998,передняя,комфорт,INTERIOR-01,11,5.4,198.7,127.5


#### Контекст

Сеть автомастерских «Глушитель и Гаечный Ключ» (Muffler And Wrench) хочет разработать модель автоматической оценки стоимости восстановления автомобиля после ДТП. Эксперты тратят время на субъективные оценки, клиенты жалуются на разницу в ценах. Нужна объективная модель, предсказывающая стоимость ремонта на основе повреждений.

#### Датасет

- damage_zone — зона повреждения (3 значения: передняя/задняя/боковая часть) (object),
- car_class — класс автомобиля (4 значения: эконом/комфорт/бизнес/премиум) (object),
- detail_code — код поврежденной детали (категориальный, ~60 уникальных значений по каталогу) (object),
- car_age — возраст автомобиля в годах (int, диапазон 0–25).
- engine_volume — объем двигателя в литрах (float, диапазон 1.0–6.0).
- mileage — пробег в тысячах километров (float, диапазон 0–400).

#### Таргет

- repair_cost — стоимость ремонта в тысячах рублей (float, диапазон 15–600).

#### Бизнес-задача

Предсказывать `repair_cost` — чтобы автоматизировать оценку стоимости ремонта, снизить влияние субъективного фактора и сократить время работы эксперта.

>#### Задание 2
Проведите кодирование категориальных данных.

In [3]:
def encoding(row):
    if row['car_class'] == 'эконом':
        return 0
    elif row['car_class'] == 'комфорт':
        return 1
    elif row['car_class'] == 'бизнес':
        return 2
    elif row['car_class'] == 'премиум':
        return 2
    else:
        return 3
    
df['car_class'] = df.apply(encoding, axis=1)
df

,damage_zone,car_class,detail_code,car_age,engine_volume,mileage,repair_cost
0,задняя,0,WHEEL-01,6,4.9,65.4,41.3
1,боковая,0,BUMPER-03,19,4.1,13.5,89.0
2,боковая,2,PAINT-02,23,4.0,56.4,155.3
3,задняя,0,SUSP-01,22,5.2,243.5,263.3
4,передняя,0,MOTOR-01,6,3.7,71.7,600.0
...,...,...,...,...,...,...,...
1995,задняя,1,BUMPER-03,0,5.1,244.6,136.4
1996,боковая,0,LIGHT-01,9,3.5,311.6,37.2
1997,передняя,0,BUMPER-03,1,1.7,287.6,106.7
1998,передняя,1,INTERIOR-01,11,5.4,198.7,127.5


In [4]:
df = pd.get_dummies(df, columns=['damage_zone'])
df

,car_class,detail_code,car_age,engine_volume,mileage,repair_cost,damage_zone_боковая,damage_zone_задняя,damage_zone_передняя
0,0,WHEEL-01,6,4.9,65.4,41.3,0,1,0
1,0,BUMPER-03,19,4.1,13.5,89.0,1,0,0
2,2,PAINT-02,23,4.0,56.4,155.3,1,0,0
3,0,SUSP-01,22,5.2,243.5,263.3,0,1,0
4,0,MOTOR-01,6,3.7,71.7,600.0,0,0,1
...,...,...,...,...,...,...,...,...,...
1995,1,BUMPER-03,0,5.1,244.6,136.4,0,1,0
1996,0,LIGHT-01,9,3.5,311.6,37.2,1,0,0
1997,0,BUMPER-03,1,1.7,287.6,106.7,0,0,1
1998,1,INTERIOR-01,11,5.4,198.7,127.5,0,0,1


In [5]:
street_list = df['detail_code'].unique()

def encoding(row):
    for street in street_list:
        if row['detail_code'] == street:
            d = df.loc[df['detail_code'] == street]
            return d['repair_cost'].mean()
        
df['detail_code'] = df.apply(encoding, axis=1)
df

,car_class,detail_code,car_age,engine_volume,mileage,repair_cost,damage_zone_боковая,damage_zone_задняя,damage_zone_передняя
0,0,99.455000,6,4.9,65.4,41.3,0,1,0
1,0,194.676667,19,4.1,13.5,89.0,1,0,0
2,2,89.755172,23,4.0,56.4,155.3,1,0,0
3,0,326.731034,22,5.2,243.5,263.3,0,1,0
4,0,555.576190,6,3.7,71.7,600.0,0,0,1
...,...,...,...,...,...,...,...,...,...
1995,1,194.676667,0,5.1,244.6,136.4,0,1,0
1996,0,90.254545,9,3.5,311.6,37.2,1,0,0
1997,0,194.676667,1,1.7,287.6,106.7,0,0,1
1998,1,114.354286,11,5.4,198.7,127.5,0,0,1


>#### Задание 3
Обучите регрессионную модель и вычислите ее коэффициент детерминации.

In [6]:
X = df.drop(columns=['repair_cost'])
y = df['repair_cost']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
print('r2_train =', r2_train)
print('r2_test  =', r2_test)

r2_train = 0.918616947805844
r2_test  = 0.9211314118511386


>#### Задание 4
Прочитайте файл `flexforge`.

In [7]:
df = pd.read_csv('data/flexforge.csv')
df

,location_tier,trainer_level,club_name,members_count,avg_monthly_fee,service_years,high_profit
0,Спальный район,средний,Ритм-фитнес Кунцево,1919,1843,8.6,1
1,Окраина/промзона,средний,FlexForge Mini,873,6204,20.2,1
2,Окраина/промзона,новички,Железная тяга,1470,6323,8.5,1
3,Спальный район,средний,Тонус-клуб Коломенская,2790,11968,1.5,1
4,ТЦ/бизнес-центр,средний,FlexForge Metro,1787,6420,13.9,1
...,...,...,...,...,...,...,...
1495,Спальный район,элита,Домашний фитнес,2784,3733,14.4,1
1496,Спальный район,средний,Заводной фитнес,300,6200,12.6,1
1497,Спальный район,средний,FlexForge Metro,2423,7406,21.4,1
1498,Спальный район,средний,Здоровая спина,626,9360,13.3,0


#### Контекст
Сеть фитнес-клубов «FlexForge» хочет разобраться, почему одни их филиалы приносят высокую прибыль (>12% маржи), а другие работают в убыток или около нуля. Менеджмент сети хочет построить классификатор, который на основе характеристик филиала предскажет его финансовую успешность.

#### Датасет

- location_tier — расположение филиала (3 значения: «ТЦ/бизнес-центр», «Спальный район рядом с метро», «Окраина / промзона») (object),

- trainer_level — уровень тренерского состава (4 значения: «новички», «средний», «профи», "элита") (object),

- club_name — название клуба/бренда (~65 уникальных значений: «FlexForge Premium ВДНХ», «FlexExpress Химки», «Качалка у Палыча» и т.д. Каждый бренд может иметь несколько залов) (object),

- members_count — количество действующих абонементов на конец месяца (int, 200–3000),

- avg_monthly_fee — средняя стоимость месячного абонемента в рублях (int, 1500–12000),

- service_years — сколько лет филиал работает на рынке (float, 0.1–25),

#### Таргет

- high_profit — маржа выше 12% (0) или ниже (1) (int).

#### Бизнес-задача

Построить бинарный классификатор, предсказывающий `high_profit`. Это позволит менеджменту «FlexForge» понять, какие факторы делают филиал успешным: престижная локация, квалифицированные тренеры, или «раскрученный» бренд клуба.

>#### Задание 5
Проведите кодирование категориальных данных.

In [8]:
df = pd.get_dummies(df, columns=['location_tier', 'trainer_level', 'club_name'])
df

,members_count,avg_monthly_fee,service_years,high_profit,location_tier_Окраина/промзона,location_tier_Спальный район,location_tier_ТЦ/бизнес-центр,trainer_level_новички,trainer_level_профи,trainer_level_средний,...,club_name_Лёгкая атлетика +,club_name_Олимпийские надежды,club_name_Ритм-фитнес Кунцево,club_name_Спартанец,club_name_Тонус-клуб Коломенская,club_name_Тренерский центр,club_name_Тяжёлый вес,club_name_Фитнес-планета,club_name_Фитнес-хаус,club_name_Элит-фитнес Рублёвка
0,1919,1843,8.6,1,0,1,0,0,0,1,...,0,0,1,0,0,0,0,0,0,0
1,873,6204,20.2,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1470,6323,8.5,1,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2790,11968,1.5,1,0,1,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
4,1787,6420,13.9,1,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,2784,3733,14.4,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1496,300,6200,12.6,1,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1497,2423,7406,21.4,1,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1498,626,9360,13.3,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


>#### Задание 6
Обучите бинарный классификатор на ближайших соседях и вычислите его метрики прогнозирования.

In [9]:
X = df.drop(columns='high_profit')
y = df['high_profit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = KNeighborsClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy_baseline = accuracy_score(y_test, y_pred)
print('accuracy_baseline =', accuracy_baseline)
precision_baseline = precision_score(y_test, y_pred)
print('precision_baseline =', precision_baseline)
recall_baseline = recall_score(y_test, y_pred)
print('recall_baseline =', recall_baseline)
f1_baseline = f1_score(y_test, y_pred)
print('f1_baseline =', f1_baseline)

accuracy_baseline = 0.936
precision_baseline = 0.958904109589041
recall_baseline = 0.9749303621169917
f1_baseline = 0.9668508287292817


>#### Задание 4
Проведите нормализацию данных. Как нормализация влияет на метрики прогнозирования?

In [10]:
scaler = MinMaxScaler()
scaler.fit(X_train)

X_train_min_max = scaler.transform(X_train)
X_test_min_max = scaler.transform(X_test)

model = KNeighborsClassifier()
model.fit(X_train_min_max, y_train)

y_pred = model.predict(X_test_min_max)

accuracy_min_max = accuracy_score(y_test, y_pred)
print('accuracy_min_max =', accuracy_min_max)
precision_min_max = precision_score(y_test, y_pred)
print('precision_min_max =', precision_min_max)
recall_min_max = recall_score(y_test, y_pred)
print('recall_min_max =', recall_min_max)
f1_min_max = f1_score(y_test, y_pred)
print('f1_min_max =', f1_min_max)

accuracy_min_max = 0.9493333333333334
precision_min_max = 0.9594594594594594
recall_min_max = 0.9888579387186629
f1_min_max = 0.9739368998628257


In [11]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_min_max = scaler.transform(X_train)
X_test_min_max = scaler.transform(X_test)

model = KNeighborsClassifier()
model.fit(X_train_min_max, y_train)

y_pred = model.predict(X_test_min_max)

accuracy_stand = accuracy_score(y_test, y_pred)
print('accuracy_stand =', accuracy_stand)
precision_stand = precision_score(y_test, y_pred)
print('precision_stand =', precision_stand)
recall_stand = recall_score(y_test, y_pred)
print('recall_stand =', recall_stand)
f1_stand = f1_score(y_test, y_pred)
print('f1_stand =', f1_stand)

accuracy_stand = 0.944
precision_stand = 0.9567567567567568
recall_stand = 0.9860724233983287
f1_stand = 0.9711934156378601


In [12]:
scaler = RobustScaler()
scaler.fit(X_train)

X_train_robust = scaler.transform(X_train)
X_test_robust = scaler.transform(X_test)

model = KNeighborsClassifier()
model.fit(X_train_robust, y_train)

y_pred = model.predict(X_test_robust)

accuracy_robust = accuracy_score(y_test, y_pred)
print('accuracy_robust =', accuracy_robust)
precision_robust = precision_score(y_test, y_pred)
print('precision_robust =', precision_robust)
recall_robust = recall_score(y_test, y_pred)
print('recall_robust =', recall_robust)
f1_robust = f1_score(y_test, y_pred)
print('f1_robust =', f1_robust)

accuracy_robust = 0.944
precision_robust = 0.9592391304347826
recall_robust = 0.9832869080779945
f1_robust = 0.9711141678129299


## Домашнее задание

#### Задание 1
Прочитайте файл `dentalpro.csv`.

In [13]:
df = pd.read_csv('data/dentalpro.csv')
df

,doctor_specialization,insurance_type,visit_duration_min,pain_level_before,waiting_days,satisfaction_score,is_promoter
0,хирург,ДМС,72.9,6.5,6,1.7,0
1,гигиенист,ДМС,98.6,1.7,6,3.2,0
2,ортодонт,без страховки,93.6,8.7,1,3.1,0
3,хирург,ДМС,26.9,6.1,13,0.5,0
4,терапевт,ДМС,26.4,1.6,4,4.6,0
...,...,...,...,...,...,...,...
1995,ортодонт,ОМС+ДМС,85.3,2.7,10,3.4,0
1996,гигиенист,ДМС,65.4,2.1,11,2.6,0
1997,терапевт,ДМС,94.2,4.6,15,2.5,0
1998,терапевт,ОМС+ДМС,63.4,9.1,3,3.4,0


#### Контекст

Сеть частных стоматологических клиник «DentalPro» хочет прогнозировать, будет ли пациент рекомендовать клинику друзьям (показатель лояльности NPS — от 0 до 10). Для упрощения решили сделать бинарную классификацию: «промоутер» (поставил 9–10 баллов) vs «не промоутер» (0–8).

#### Датасет

- doctor_specialization — специализация лечащего врача (4 значения: «терапевт», «хирург», «ортодонт», «гигиенист») (object),
- insurance_type — тип страховки (3 значения: «ДМС», «полис ОМС+ДМС», «без страховки») (object),
- visit_duration_min — продолжительность приёма в минутах (float, 10–120),
- pain_level_before — уровень боли до приёма по шкале от 0 до 10 (float),
- waiting_days — количество дней ожидания приёма (int, 0–30),
- satisfaction_score — оценка CSAT (Customer Satisfaction Score) удовлетворённости после приёма (float, 0–10). 

#### Таргет

- is_promoter — пациент поставил 9–10 баллов NPS (Net Promoter Score) (1 — промоутер, 0 — не промоутер).

#### Бизнес-задача

Построить бинарный классификатор, чтобы понимать, какие факторы превращают пациента в промоутера, и можно ли это предсказывать до того, как он поставит оценку (до заполнения анкеты удовлетворённости).

>#### Задание 2
Проведите бинарную классификацию. Добейтесь достаточно высоких метрик.

In [14]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1796 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   doctor_specialization  1796 non-null   object 
 1   insurance_type         1796 non-null   object 
 2   visit_duration_min     1796 non-null   float64
 3   pain_level_before      1796 non-null   float64
 4   waiting_days           1796 non-null   int64  
 5   satisfaction_score     1796 non-null   float64
 6   is_promoter            1796 non-null   int64  
dtypes: float64(3), int64(2), object(2)
memory usage: 112.2+ KB


In [15]:
df = pd.get_dummies(df, columns=['doctor_specialization', 'insurance_type'])
df

,visit_duration_min,pain_level_before,waiting_days,satisfaction_score,is_promoter,doctor_specialization_гигиенист,doctor_specialization_ортодонт,doctor_specialization_терапевт,doctor_specialization_хирург,insurance_type_ДМС,insurance_type_ОМС+ДМС,insurance_type_без страховки
0,72.9,6.5,6,1.7,0,0,0,0,1,1,0,0
1,98.6,1.7,6,3.2,0,1,0,0,0,1,0,0
2,93.6,8.7,1,3.1,0,0,1,0,0,0,0,1
3,26.9,6.1,13,0.5,0,0,0,0,1,1,0,0
4,26.4,1.6,4,4.6,0,0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1995,85.3,2.7,10,3.4,0,0,1,0,0,0,1,0
1996,65.4,2.1,11,2.6,0,1,0,0,0,1,0,0
1997,94.2,4.6,15,2.5,0,0,0,1,0,1,0,0
1998,63.4,9.1,3,3.4,0,0,0,1,0,0,1,0


In [16]:
X = df.drop(columns='is_promoter')
y = df['is_promoter']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = KNeighborsClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy_baseline = accuracy_score(y_test, y_pred)
print('accuracy_baseline =', accuracy_baseline)
precision_baseline = precision_score(y_test, y_pred)
print('precision_baseline =', precision_baseline)
recall_baseline = recall_score(y_test, y_pred)
print('recall_baseline =', recall_baseline)
f1_baseline = f1_score(y_test, y_pred)
print('f1_baseline =', f1_baseline)

accuracy_baseline = 0.9955456570155902
precision_baseline = 0.0
recall_baseline = 0.0
f1_baseline = 0.0


In [17]:
scaler = RobustScaler()
scaler.fit(X_train)

X_train_robust = scaler.transform(X_train)
X_test_robust = scaler.transform(X_test)

model = KNeighborsClassifier()
model.fit(X_train_robust, y_train)

y_pred = model.predict(X_test_robust)

accuracy_robust = accuracy_score(y_test, y_pred)
print('accuracy_robust =', accuracy_robust)
precision_robust = precision_score(y_test, y_pred)
print('precision_robust =', precision_robust)
recall_robust = recall_score(y_test, y_pred)
print('recall_robust =', recall_robust)
f1_robust = f1_score(y_test, y_pred)
print('f1_robust =', f1_robust)

accuracy_robust = 0.9933184855233853
precision_robust = 0.3333333333333333
recall_robust = 0.5
f1_robust = 0.4


In [18]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_stand = scaler.transform(X_train)
X_test_stand = scaler.transform(X_test)

model = KNeighborsClassifier()
model.fit(X_train_stand, y_train)

y_pred = model.predict(X_test_stand)

accuracy_stand = accuracy_score(y_test, y_pred)
print('accuracy_stand =', accuracy_stand)
precision_stand = precision_score(y_test, y_pred)
print('precision_stand =', precision_stand)
recall_stand = recall_score(y_test, y_pred)
print('recall_stand =', recall_stand)
f1_stand = f1_score(y_test, y_pred)
print('f1_stand =', f1_stand)

accuracy_stand = 0.9910913140311804
precision_stand = 0.0
recall_stand = 0.0
f1_stand = 0.0


In [19]:
scaler = MinMaxScaler()
scaler.fit(X_train)

X_train_min_max = scaler.transform(X_train)
X_test_min_max = scaler.transform(X_test)

model = KNeighborsClassifier()
model.fit(X_train_min_max, y_train)

y_pred = model.predict(X_test_min_max)

accuracy_min_max = accuracy_score(y_test, y_pred)
print('accuracy_min_max =', accuracy_min_max)
precision_min_max = precision_score(y_test, y_pred)
print('precision_min_max =', precision_min_max)
recall_min_max = recall_score(y_test, y_pred)
print('recall_min_max =', recall_min_max)
f1_min_max = f1_score(y_test, y_pred)
print('f1_min_max =', f1_min_max)

accuracy_min_max = 0.9910913140311804
precision_min_max = 0.0
recall_min_max = 0.0
f1_min_max = 0.0
